In [2]:

!pip install langchain langchain-openai langchain-community langgraph langchain-groq openai requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 37.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google

In [3]:
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
import requests
import json
import os

print("All imports successful")

All imports successful


In [13]:
import os
os.environ["GROQ_API_KEY"] = "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"

from google.colab import files
uploaded = files.upload()


Saving places.json to places (1).json
Saving hotels.json to hotels (1).json
Saving flights.json to flights (1).json


In [14]:
with open("flights.json") as f:
  flights_data = json.load(f)
with open("hotels.json") as f:
  hotels_data = json.load(f)
with open("places.json") as f:
  places_data = json.load(f)

print(f"Flights: {len(flights_data)} | Hotels: {len(hotels_data)} | Places: {len(places_data)}")

Flights: 30 | Hotels: 40 | Places: 40


In [15]:
@tool
def search_flights(source: str, destination: str) -> str:
    """
    Search for available flights between two cities.
    Args:
        source: departure city name (e.g. 'Mumbai')
        destination: arrival city name (e.g. 'Goa')
    Returns cheapest available flight.
    """
    try:
        results = [
            f for f in flights_data
            if f["from"].lower() == source.lower()
            and f["to"].lower() == destination.lower()
        ]
        if not results:
            return f"No flights found from {source} to {destination}."
        cheapest = sorted(results, key=lambda x: x["price"])[0]
        return (
            f"Airline: {cheapest['airline']} | "
            f"Price: {cheapest['price']} | "
            f"Departure: {cheapest['departure_time']} | "
            f"Arrival: {cheapest['arrival_time']}"
        )
    except Exception as e:
        return f"Error searching flights: {str(e)}"
@tool
def search_hotels(city: str, max_price: int = 99999) -> str:
    """
    Search for best available hotel in a city.
    Args:
        city: city name (e.g. 'Goa')
        max_price: maximum price per night in rupees (default: 99999)
    Returns highest rated hotel within budget.
    """
    try:
        results = [
            h for h in hotels_data
            if h["city"].lower() == city.lower()
            and h["price_per_night"] <= max_price
        ]
        if not results:
            return f"No hotels found in {city} under {max_price} per night."
        best = sorted(results, key=lambda x: x["stars"], reverse=True)[0]
        return (
            f"Hotel: {best['name']} | "
            f"Stars: {best['stars']} | "
            f"Price: {best['price_per_night']} per night | "
            f"Amenities: {', '.join(best['amenities'])}"
        )
    except Exception as e:
        return f"Error searching hotels: {str(e)}"
@tool
def search_places(city: str, num_days: int) -> str:
    """
    Search top attractions and places to visit in a city.
    Args:
        city: city name (e.g. 'Goa')
        num_days: number of days for the trip (e.g. 3)
    Returns day-wise itinerary of top rated places.
    """
    try:
        results = [
            p for p in places_data
            if p["city"].lower() == city.lower()
        ]
        if not results:
            return f"No places found in {city}."
        top_places = sorted(results, key=lambda x: x["rating"], reverse=True)
        top_places = top_places[:num_days * 2]
        itinerary = ""
        for day in range(num_days):
            day_places = top_places[day*2:(day+1)*2]
            place_names = [
                f"{p['name']} ({p['type']}, {p['rating']} stars)"
                for p in day_places
            ]
            itinerary += f"Day {day+1}: {' | '.join(place_names)}\n"
        return itinerary.strip()
    except Exception as e:
        return f"Error searching places: {str(e)}"
@tool
def get_weather(city: str) -> str:
    """
    Get 7-day weather forecast for a city using Open-Meteo API.
    Args:
        city: city name (e.g. 'Goa')
    Returns day-wise temperature forecast.
    """
    CITY_COORDS = {
        "delhi": (28.6139, 77.2090),
        "mumbai": (19.0760, 72.8777),
        "goa": (15.2993, 74.1240),
        "bangalore": (12.9716, 77.5946),
        "hyderabad": (17.3850, 78.4867),
        "chennai": (13.0827, 80.2707),
        "kolkata": (22.5726, 88.3639),
        "jaipur": (26.9124, 75.7873)
    }
    try:
        city_lower = city.strip().lower()
        if city_lower not in CITY_COORDS:
            return f"Weather not available for {city}. Available cities: {', '.join(CITY_COORDS.keys())}"
        lat, lon = CITY_COORDS[city_lower]
        url = (
            f"https://api.open-meteo.com/v1/forecast"
            f"?latitude={lat}&longitude={lon}"
            f"&daily=temperature_2m_max,temperature_2m_min"
            f"&timezone=auto&forecast_days=7"
        )
        response = requests.get(url, timeout=10)
        data = response.json()
        dates = data["daily"]["time"][:7]
        max_temps = data["daily"]["temperature_2m_max"][:7]
        min_temps = data["daily"]["temperature_2m_min"][:7]
        forecast = ""
        for date, max_t, min_t in zip(dates, max_temps, min_temps):
            forecast += f"{date}: Max {max_t}C | Min {min_t}C\n"
        return f"Weather for {city.title()}:\n{forecast.strip()}"
    except requests.exceptions.Timeout:
        return "Weather API timed out. Please try again."
    except Exception as e:
        return f"Error fetching weather: {str(e)}"
@tool
def calculate_budget(flight_cost: int, hotel_per_night: int, num_nights: int, daily_expenses: int = 800) -> str:
    """
    Calculate total trip budget breakdown.
    Args:
        flight_cost: flight price in rupees (use exact number from search_flights result)
        hotel_per_night: hotel price per night in rupees (use exact number from search_hotels result)
        num_nights: number of nights staying
        daily_expenses: estimated daily food and local travel cost (default: 800)
    Returns complete budget breakdown with total.
    """
    try:
        hotel_total = hotel_per_night * num_nights
        food_total = daily_expenses * num_nights
        grand_total = flight_cost + hotel_total + food_total
        return (
            f"Budget Breakdown:\n"
            f"  Flight:                  {flight_cost}\n"
            f"  Hotel ({num_nights} nights): {hotel_total}\n"
            f"  Food and Local Travel:   {food_total}\n"
            f"  ---------------------------------\n"
            f"  Total:                   {grand_total}"
        )
    except Exception as e:
        return f"Error calculating budget: {str(e)}"
print("All 5 tools defined successfully")
print(search_flights.invoke({"source": "Mumbai", "destination": "Goa"}))
print(search_hotels.invoke({"city": "Goa"}))
print(get_weather.invoke({"city": "Goa"}))

All 5 tools defined successfully
Airline: SpiceJet | Price: 3304 | Departure: 2025-07-15T14:38:00 | Arrival: 2025-07-15T18:38:00
Hotel: Comfort Suites | Stars: 5 | Price: 2828 per night | Amenities: spa, pool, wifi, gym
Weather for Goa:
2026-05-26: Max 34.2C | Min 26.1C
2026-05-27: Max 32.9C | Min 26.4C
2026-05-28: Max 32.7C | Min 26.1C
2026-05-29: Max 31.6C | Min 26.1C
2026-05-30: Max 32.0C | Min 25.7C
2026-05-31: Max 31.5C | Min 26.3C
2026-06-01: Max 32.2C | Min 25.6C


In [16]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key="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"
)
tools = [search_flights, search_hotels, search_places, get_weather, calculate_budget]
system_prompt = """You are an expert AI travel planning assistant for Indian cities.
Available cities: Delhi, Mumbai, Goa, Bangalore, Hyderabad, Chennai, Kolkata, Jaipur
You must follow these steps in exact order for every trip planning request:
Step 1: Call search_flights with the source and destination city names
Step 2: Call search_hotels with the destination city name
Step 3: Call get_weather with the destination city name
Step 4: Call search_places with the destination city name and number of days
Step 5: Call calculate_budget using the EXACT price numbers returned by search_flights and search_hotels
Critical rules:
- Never use 0 as flight_cost or hotel_per_night in calculate_budget
- Always extract the exact numeric price from tool results
- If a flight result says 'Price: 3304', use 3304 as flight_cost
- If a hotel result says 'Price: 2828 per night', use 2828 as hotel_per_night
- Never fabricate data. Only use what tools return.
End your response with a clean structured itinerary covering:
- Flight details
- Hotel details
- Day-wise places to visit
- Weather summary
- Budget breakdown"""
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)
print("Agent created successfully")

Agent created successfully


/tmp/ipykernel_5564/545839970.py:27: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [17]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Plan a 3-day trip to Goa from Mumbai. My budget is 15000."
    }]
})
print("=" * 60)
print("FINAL ITINERARY")
print("=" * 60)
print(result["messages"][-1].content)

FINAL ITINERARY
Here's your 3-day trip plan to Goa from Mumbai:

**Flight Details:** 
- Airline: SpiceJet
- Price: 3304
- Departure: 2025-07-15T14:38:00
- Arrival: 2025-07-15T18:38:00

**Hotel Details:** 
- Hotel: Comfort Suites
- Stars: 5
- Price: 2828 per night
- Amenities: spa, pool, wifi, gym

**Day-wise Places to Visit:** 
Day 1: Famous Park (museum, 4.5 stars) | Beautiful Park (fort, 4.3 stars)
Day 2: Popular Lake (museum, 4.2 stars) | Historic Park (fort, 4.1 stars)
Day 3: Beautiful Lake (market, 4.0 stars)

**Weather Summary:** 
2026-05-26: Max 34.2C | Min 26.1C
2026-05-27: Max 32.9C | Min 26.4C
2026-05-28: Max 32.7C | Min 26.1C
2026-05-29: Max 31.6C | Min 26.1C
2026-05-30: Max 32.0C | Min 25.7C
2026-05-31: Max 31.5C | Min 26.3C
2026-06-01: Max 32.2C | Min 25.6C

**Budget Breakdown:** 
  Flight:                  3304
  Hotel (3 nights): 8484
  Food and Local Travel:   2400
  ---------------------------------
  Total:                   14188

This trip plan should fit your budge